In [307]:
def rule(header):
    """Separator for a '|'-joined header: '+' under every '|', '-' everywhere else."""
    start = max(len(header) - len(header.lstrip()), 0)
    return " " * start + "".join("+" if c == "|" else "-" for c in header[start:])
def table_header(header):
    print(header)
    print(rule(header))

def header(columns, debug=False):
    """Header block for an ASCII table: the label rows, then the rule.

    A column is a string, or a tuple of 2+ lines whose extra rows are centered
    under the first. Column width = the widest of that column's own lines.
    debug=True prepends a row showing each column's computed width."""
    cols = [(c,) if isinstance(c, str) else tuple(c) for c in columns]
    widths = [max(map(len, c)) for c in cols]

    lines = []
    for r in range(max(map(len, cols))):
        cells = []
        for col, w in zip(cols, widths):
            text = col[r] if r < len(col) else ""
            cells.append(text.ljust(w) if r == 0 else text.center(w))
        lines.append(" | ".join(cells))

    rule = "".join("+" if ch == "|" else "-" for ch in lines[0])
    if debug:
        lines.insert(0, " | ".join(str(w).center(w) for w in widths))
    lines.append(rule)
    print("\n".join(line.rstrip() for line in lines))


In [308]:
import numpy as np

In [309]:
class RandomWalk:
    """19-state random walk. Fixed uniform-random policy => PREDICTION, like ex 3.
    step() ignores its action, and true_V() solves the MDP exactly (no sampling)."""

    N = 19

    def __init__(self, rng=None):
        self.rng = rng
        self.s = None
        self.nA = 2
        self.P = self._build_model()

    def reset(self):
        self.s = (self.N + 1) // 2                      # start in the middle
        return self.s

    def _move(self, s: int, a: int):
        s += [-1, 1][a]
        return s   

    def _build_model(self) -> dict:
        P = {s: {a: [] for a in range(self.nA)} for s in range(self.N)}
        for s in range(self.N):
            for a in range(self.nA):
                outcomes = {0: 0.5, 1: 0.5}
                agg: dict[int, list] = {}    # next_state -> [prob, reward, done]
                for act, prob in outcomes.items():
                    nxt = self._move(s, act)
                    done = nxt in (-1, self.N)
                    reward = 1.0 if nxt == self.N else 0.0
                    if nxt in agg:            
                        agg[nxt][0] += prob
                    else:
                        agg[nxt] = [prob, reward, done]
                P[s][a] = [(p, ns, r, d) for ns, (p, r, d) in agg.items()]
        return P          

    def step(self, _a=None):
        self.s += int(self.rng.choice([-1, 1]))
        if self.s == 0:
            return self.s, 0.0, True
        if self.s == self.N + 1:
            return self.s, 1.0, True
        return self.s, 0.0, False

    def true_V(self, gamma):
        """Exact V^pi for states 1..N by linear solve (ground truth for grading)."""
        N = self.N
        P = np.zeros((N, N))                            # non-terminal transitions
        r = np.zeros(N)
        for i, s in enumerate(range(1, N + 1)):
            for ns in (s - 1, s + 1):
                if ns == 0:
                    continue                            # left end: reward 0, absorbing
                if ns == N + 1:
                    r[i] += 0.5 * 1.0                   # right end: reward +1
                    continue
                P[i, ns - 1] += 0.5
        return np.linalg.solve(np.eye(N) - gamma * P, r)

In [310]:
seed = 0
rng = np.random.default_rng(seed)
env = RandomWalk()
V = env.true_V(gamma=1.0)
V

array([0.05, 0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 , 0.55,
       0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95])

In [311]:
print(
    f"  check the exact solve against the closed form s/20: max diff "
    f"{np.abs(V - np.arange(1, 20) / 20).max():.2e}"
)

  check the exact solve against the closed form s/20: max diff 2.22e-16


# Derivation for s/20


Step 1: at gamma=1, "value" here means "probability of winning"
---------------------------------------------------------------

Every episode ends one of two ways: you reach 20 and collect **+1**, or you reach 0 and collect **0**. Nothing else ever pays. And `gamma=1` means no shrinking, so the return of an episode is literally 1 or 0.

`V(s)` is the _average_ return over many episodes from `s`. Averaging a bunch of 1s and 0s just gives you the fraction that were 1s:

    V(s) = probability you reach 20 before you reach 0, starting from s
    

Quick sanity check: state 10 is dead center, and the walk is 50/50, so it's a coin flip — `V(10)` should be 0.5. Your printout: `0.5`. Good.

Step 2: the one rule the walk gives you
---------------------------------------

Stand on state `s`. You flip a fair coin: half the time you step to `s-1`, half to `s+1`. After that step, your chance of winning is whatever it is _from there_. So:

    chance of winning from s  =  half of (chance from s-1)  +  half of (chance from s+1)
    
    V(s) = ( V(s-1) + V(s+1) ) / 2
    

In words: **every state's value is the average of its two neighbours.** Check it on your numbers at s=10:

    ( V(9) + V(11) ) / 2  =  ( 0.45 + 0.55 ) / 2  =  0.50  =  V(10)   ✓
    

Try it anywhere else and it holds. That's the whole physics of this problem.

Step 3: what does "every point is the average of its neighbours" force?
-----------------------------------------------------------------------

This is the only step with any real content, so let's take it slowly. Multiply that equation by 2:

    2 V(s) = V(s-1) + V(s+1)
    

Now split the left side into `V(s) + V(s)` and move one of each to the other side:

    V(s) - V(s-1)  =  V(s+1) - V(s)
       ^^^                ^^^
       the step you just took     the step you're about to take
    

Read that in plain English: **the gap between `s-1` and `s` is the same as the gap between `s` and `s+1`.** Every gap equals the next gap. So every gap in the whole chain is the same number.

A sequence where every step up is the same size is a straight line. Look at your actual values:

    V:      0.05  0.10  0.15  0.20  0.25  ...  0.90  0.95
    gaps:      0.05  0.05  0.05  0.05      ...     0.05
    

All gaps 0.05. Not approximately — exactly.

Step 4: the two ends fix which straight line
--------------------------------------------

We know two values for free, no math needed:

*   `V(0) = 0` — you've already lost, probability of winning is 0.
*   `V(20) = 1` — you've already won.

So: start at 0, climb to 1, in **20 equal steps** (from 0 to 20). Each step must be `1/20 = 0.05`. After `s` steps you're at `0.05 * s`:

    V(s) = s/20
    

That's it. `V(7) = 7/20 = 0.35`. Check the array: `0.35`. ✓

The 20 is the distance between the two ends (0 and 20), not the 19 interior states — that's the only place people usually slip.

# New

In [312]:
def q_from_v(
    env: RandomWalk,
    V: np.ndarray,
    s: int,
    gamma: float,
):
    q = np.zeros(env.nA)
    for a in range(env.nA):
        for prob, ns, r, done in env.P[s][a]:
            q[a] += prob * (r + gamma * (V[ns] if not done else 0))
    return q

def value_iteration(
    env: RandomWalk,
    gamma=0.9,
    theta=1e-6,
    max_iters=10000,
):
    V = np.zeros(env.N)
    delta = float('inf')
    i = 0
    while delta >= theta and i < max_iters:
        delta = 0.0
        for s in range(env.N):
            v_old = V[s]
            V[s] = np.max(q_from_v(env, V, s, gamma))
            delta = max(delta, abs(V[s] - v_old))
        i += 1
    converged = delta < theta
    return V, converged

In [313]:
V_calc, converged = value_iteration(env, gamma=1.0)
np.round(V_calc, 2)

array([0.05, 0.1 , 0.15, 0.2 , 0.25, 0.3 , 0.35, 0.4 , 0.45, 0.5 , 0.55,
       0.6 , 0.65, 0.7 , 0.75, 0.8 , 0.85, 0.9 , 0.95])

In [314]:
import sys; sys.exit()

SystemExit: 

In [ ]:
def make_phi(kind, N=RandomWalk.N):
    """Feature MATRIX (N, d): row i is phi(s) for state s = i+1. v(s;w) = phi(s) . w.
    The choice of phi is the choice of how much states share — the whole ballgame."""
    x = np.arange(1, N + 1) / (N + 1)                   # normalized position in (0,1)
    if kind == "const":
        return np.ones((N, 1))                          # one weight for the whole world
    if kind.startswith("poly"):
        deg = int(kind[4:])
        return np.stack([x ** k for k in range(deg + 1)], axis=1)
    if kind == "agg5":                                  # state aggregation: 5 groups
        g = np.minimum((np.arange(N) // 4), 4)
        return np.eye(5)[g]
    if kind == "rbf5":                                  # 5 gaussian bumps, width 0.125
        c = np.linspace(0.1, 0.9, 5)
        return np.exp(-((x[:, None] - c[None, :]) ** 2) / (2 * 0.125 ** 2))
    if kind == "onehot":
        return np.eye(N)                                # == the TABLE (layer 4)
    raise ValueError(kind)

In [ ]:
print("\n  phi(s) = [1, s/20]  ->  2 weights for 19 states.")
Phi = make_phi("poly1")
Phi


  phi(s) = [1, s/20]  ->  2 weights for 19 states.


array([[1.  , 0.05],
       [1.  , 0.1 ],
       [1.  , 0.15],
       [1.  , 0.2 ],
       [1.  , 0.25],
       [1.  , 0.3 ],
       [1.  , 0.35],
       [1.  , 0.4 ],
       [1.  , 0.45],
       [1.  , 0.5 ],
       [1.  , 0.55],
       [1.  , 0.6 ],
       [1.  , 0.65],
       [1.  , 0.7 ],
       [1.  , 0.75],
       [1.  , 0.8 ],
       [1.  , 0.85],
       [1.  , 0.9 ],
       [1.  , 0.95]])

In [ ]:
print(f'{Phi.shape=}')
print(f'{len(Phi)=}')

Phi.shape=(19, 2)
len(Phi)=19


In [ ]:
def _rms(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

In [ ]:
print("   SGD steps |    w = [bias, slope]    | RMS vs true V")
print("  -----------+-------------------------+---------------")
w = np.zeros(2)
prev = 0
for n in (10, 100, 1000, 10000, 100000):
    for _ in range(n - prev):                       # keep fitting the same w
        s = int(rng.integers(len(Phi)))
        w += 0.05 * (V[s] - Phi[s] @ w) * Phi[s]    # sgd_fit's step, constant alpha
    prev = n
    print(f"  {n:9,d}  |  [{w[0]:+.4f}, {w[1]:+.4f}]     |    {_rms(Phi @ w, V):.5f}")

   SGD steps |    w = [bias, slope]    | RMS vs true V
  -----------+-------------------------+---------------
         10  |  [+0.1392, +0.0965]     |    0.39867
        100  |  [+0.3047, +0.4063]     |    0.16279
      1,000  |  [+0.0230, +0.9586]     |    0.01157
     10,000  |  [+0.0000, +1.0000]     |    0.00000
    100,000  |  [+0.0000, +1.0000]     |    0.00000


`V[s] - Phi[s] @ w` is the **error** (also called the residual, or in RL the TD error `delta`). The **loss** is that error _squared_. That distinction is exactly why `Phi[s]` shows up at the end of the line.

Where the line comes from
-------------------------

Write the loss for one state — half the squared error (the ½ is just so the 2 cancels later):

    v     = Phi[s] @ w             <- our prediction at state s
    err   = V[s] - v               <- how wrong we are  (your "loss")
    L(w)  = 0.5 * err^2            <- the actual loss
    

Now differentiate `L` with respect to `w`, one piece at a time:

    dL/d(err)  = err                       (derivative of 0.5*err^2)
    d(err)/dv  = -1                        (err = V[s] - v, and V[s] is a constant)
    dv/dw      = Phi[s]                    (v = Phi[s] @ w is linear in w)
    
    chain them:  dL/dw = err * (-1) * Phi[s] = -err * Phi[s]
    

Gradient descent says step _against_ the gradient:

    w  =  w - lr * dL/dw
       =  w - lr * ( -err * Phi[s] )
       =  w + lr * err * Phi[s]
    

which is your line, character for character:

    w += 0.05 * (V[s] - Phi[s] @ w) * Phi[s]
    #     lr        error (delta)       gradient of v w.r.t. w
    

So yes, `0.05` is the learning rate. The other two pieces are **how wrong you are** and **who's responsible for it**.

What `Phi[s]` is doing there
----------------------------

`w` isn't the value — it's the _recipe_ for computing values. When the prediction is too low, you need to know _which weights to raise_. `dv/dw = Phi[s]` answers that: a feature that was large at this state had a big influence on the prediction, so it gets a big share of the correction; a feature that was **zero** here had no influence and gets **no** update at all.

Concretely, at state 5 with `poly1`, `Phi[5] = [1, 0.25]` (bias, then `x = 5/20`), true `V = 0.25`, starting from `w = [0, 0]`:

    v    = [1, 0.25] @ [0, 0]      = 0
    err  = 0.25 - 0                = 0.25
    w   += 0.05 * 0.25 * [1, 0.25] = [0.0125, 0.003125]
    

The bias weight moved 4x more than the slope weight — because the bias feature is 1 here and the slope feature is only 0.25. Check the prediction improved:

    v_new = [1, 0.25] @ [0.0125, 0.003125] = 0.01328
    err   : 0.25  ->  0.2367
    

And that "no influence ⟹ no update" rule is the whole Layer 2B table in one sentence: with **one-hot** features, `Phi[s]` is zero everywhere except position `s`, so only one weight ever moves — that's the tabular update `V[s] += lr * err`. Your line _contains_ the tabular rule as a special case.

One bonus that falls out for free
---------------------------------

Notice the error shrank by a fixed fraction. In general:

    err_new = err * ( 1 - lr * (Phi[s] @ Phi[s]) )
    

Here: `1 - 0.05 * (1² + 0.25²) = 0.9469`, and `0.25 * 0.9469 = 0.2367` ✓.

Each visit multiplies the error at that state by that factor, so it decays geometrically — and if `lr` were bigger than `2 / (Phi[s] @ Phi[s])` the factor would be less than −1 and the error would _grow_ every step. That's the real reason learning rates blow up, and the reason people normalize features: `||phi||²` sets your safe learning-rate range.